# Newberry 3-D superhot demonstration: 400 °C

This notebook uses only geoPFA's public, configuration-driven workflow. The configuration is visible in the notebook; the processed JSON/layer tree produced by the standard Newberry workflow; local data and generated outputs are not version-controlled. The probabilistic target map is compared with the traditional `VoterVeto` workflow.

The heat term is the thermal-model exceedance probability `P(T > 400 °C)` and is deliberately **not** updated with available 150 °C proxy labels. Reservoir and insulation evidence coefficients are propagated from explicit priors because target-matched labels do not support posterior updating.

## Load processed inputs and declare the model


In [ ]:
import copy
import os
from pathlib import Path

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

from geopfa.layer_combination import VoterVeto
from geopfa.prob import ProbabilisticConfig, run_probabilistic
from geopfa.prob import load_processed_pfa

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise FileNotFoundError("geoPFA repository root not found")


def required_local_path(env_name: str, default: Path) -> Path:
    path = Path(os.environ.get(env_name, default)).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(
            f"Required local input is absent: {path}. Set {env_name} or prepare "
            "the documented public study data before running this notebook."
        )
    return path

repo_root = find_repo_root(Path.cwd())
project_dir = repo_root / "examples" / "Newberry" / "3D"
output_root = Path(
    os.environ.get("GEOPFA_DEMO_OUTPUT_ROOT", project_dir / "outputs")
).expanduser().resolve()
output_dir = output_root / "newberry_superhot_400c"
processed_config_path = required_local_path(
    "GEOPFA_NEWBERRY_PROCESSED_CONFIG", project_dir / "config" / "newberry_superhot_processed_config.json"
)
processed_data_dir = required_local_path(
    "GEOPFA_NEWBERRY_PROCESSED_DATA", project_dir / "data"
)
pfa, input_artifacts = load_processed_pfa(
    processed_config_path,
    processed_data_dir,
    crs="EPSG:26910",
)

config_dict = {'enabled': True,
 'output_dir': '../outputs/superhot_400c',
 'dimensions': '3d',
 'labels': {'observation_models': {'heat': {'family': 'gaussian'}}},
 'alpha': {'heat': {'mode': 'thermal_layer_exceedance',
                    'layer': 'temperature_model_500m',
                    'threshold': 400.0,
                    'uncertainty_column': 'temperature_predictive_sd_c',
                    'p_min': 1e-12,
                    'p_max': 0.999999999999,
                    'force_prior_predictive': True,
                    'use_evidence_prior': False},
           'reservoir': {'mode': 'scalar',
                         'scalar_fallback_pr0': 0.5,
                         'force_prior_predictive': True,
                         'use_evidence_prior': True},
           'insulation': {'mode': 'scalar',
                          'scalar_fallback_pr0': 0.5,
                          'force_prior_predictive': True,
                          'use_evidence_prior': True}},
 'evidence': {'standardization': 'prediction_support',
              'include_layers': ['density_joint_inv',
                                 'mt_resistivity_joint_inv',
                                 'temperature_model_500m',
                                 'earthquakes',
                                 'ring_faults',
                                 'lineaments'],
              'regularization': {'prior_means': {'reservoir:density_joint_inv': 0.6931471805599453,
                                                 'reservoir:mt_resistivity_joint_inv': 0.6931471805599453,
                                                 'reservoir:earthquakes': 0.6931471805599453,
                                                 'reservoir:ring_faults': 0.6931471805599453,
                                                 'reservoir:lineaments': 0.6931471805599453,
                                                 'insulation:density_joint_inv': 0.6931471805599453,
                                                 'insulation:mt_resistivity_joint_inv': 0.6931471805599453,
                                                 'insulation:earthquakes': 0.6931471805599453,
                                                 'insulation:temperature_model_500m': 0.6931471805599453},
                                 'prior_precisions': {'reservoir:density_joint_inv': 4.0,
                                                      'reservoir:mt_resistivity_joint_inv': 4.0,
                                                      'reservoir:earthquakes': 4.0,
                                                      'reservoir:ring_faults': 4.0,
                                                      'reservoir:lineaments': 4.0,
                                                      'insulation:density_joint_inv': 4.0,
                                                      'insulation:mt_resistivity_joint_inv': 4.0,
                                                      'insulation:earthquakes': 4.0,
                                                      'insulation:temperature_model_500m': 4.0}}},
 'spatial_field': {'enabled': False},
 'inference': {'backend': 'gblk',
               'gblk_bayesian': {'enabled': True,
                                 'n_draws': 512,
                                 'seed': 20260909,
                                 'ci_level': 0.95,
                                 'cluster_effect': False}},
 'calibration': {'method': 'none'},
 'combination': {'rule': 'product'},
 'scenarios': [],
 'outputs': {'posterior_draw_blocks': False,
             'posterior_draw_block_size': 32,
             'format': ['parquet']}}
config_dict["output_dir"] = str(output_dir)
config = ProbabilisticConfig.from_dict(config_dict)

config


## Run both workflows


In [ ]:
pfa_vv = copy.deepcopy(pfa)
pfa_vv = VoterVeto.do_voter_veto(
    pfa_vv,
    normalize_method="minmax",
    component_veto=False,
    criteria_veto=True,
    normalize=True,
    norm_to=5,
)
vv_volume = pfa_vv["pr_norm"]
model_result = run_probabilistic(
    pfa,
    config,
    input_artifacts=input_artifacts,
)
heat_probability = model_result.components["heat"].probability
combined_probability = model_result.combined
if len(heat_probability) != 439_198:
    raise ValueError(
        f"expected 439,198 supported Newberry cells, found {len(heat_probability):,}"
    )
heat_probability[["probability", "probability_lo", "probability_hi"]].describe()

## Compare the final surfaces


In [ ]:
z = heat_probability.geometry.z.to_numpy()
depth_table = heat_probability.assign(z=z).groupby("z", sort=True)["probability"].max()
display_z = float(depth_table.idxmax())
mask = np.isclose(z, display_z)
heat_slice = heat_probability.loc[mask]
combined_slice = combined_probability.loc[mask]
combined_vmax = float(combined_probability.probability.quantile(0.99))
vv_heat = vv_volume.loc[np.isclose(vv_volume.geometry.z.to_numpy(), display_z)]

def regular_surface_image(surface, value_column: str, *, name: str):
    x = surface.geometry.x.to_numpy(dtype=float)
    y = surface.geometry.y.to_numpy(dtype=float)
    values = surface[value_column].to_numpy(dtype=float)
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError(f"{name} surface coordinates must be finite")
    x_axis = np.unique(x)
    y_axis = np.unique(y)
    if x_axis.size < 2 or y_axis.size < 2:
        raise ValueError(f"{name} surface must span a two-dimensional grid")
    x_step = np.diff(x_axis)
    y_step = np.diff(y_axis)
    if not np.allclose(x_step, x_step[0]) or not np.allclose(y_step, y_step[0]):
        raise ValueError(f"{name} surface must use regular grid spacing")
    x_index = np.searchsorted(x_axis, x)
    y_index = np.searchsorted(y_axis, y)
    image = np.full((y_axis.size, x_axis.size), np.nan, dtype=float)
    if np.unique(np.ravel_multi_index((y_index, x_index), image.shape)).size != len(surface):
        raise ValueError(f"{name} surface has duplicate grid cells")
    image[y_index, x_index] = values
    extent = [
        x_axis[0] - x_step[0] / 2,
        x_axis[-1] + x_step[0] / 2,
        y_axis[0] - y_step[0] / 2,
        y_axis[-1] + y_step[0] / 2,
    ]
    return image, extent


def add_map_colorbar(image, axis, label: str) -> None:
    colorbar_axis = make_axes_locatable(axis).append_axes(
        "right", size="5%", pad=0.05
    )
    image.figure.colorbar(image, cax=colorbar_axis, label=label)


fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
vv_image, vv_extent = regular_surface_image(
    vv_heat, "favorability", name="VoterVeto"
)
heat_image, heat_extent = regular_surface_image(
    heat_slice, "probability", name="heat probability"
)
combined_image, combined_extent = regular_surface_image(
    combined_slice, "probability", name="combined probability"
)
vv_plot = axes[0].imshow(
    vv_image, extent=vv_extent, origin="lower", interpolation="bilinear",
    cmap="viridis", vmin=0, vmax=5,
)
heat_plot = axes[1].imshow(
    heat_image, extent=heat_extent, origin="lower", interpolation="bilinear",
    cmap="magma", vmin=0, vmax=1,
)
combined_plot = axes[2].imshow(
    combined_image, extent=combined_extent, origin="lower", interpolation="bilinear",
    cmap="magma", vmin=0, vmax=combined_vmax,
)
add_map_colorbar(vv_plot, axes[0], "Ordinal score")
add_map_colorbar(heat_plot, axes[1], "Probability")
add_map_colorbar(combined_plot, axes[2], "Probability")
axes[0].set_title("Traditional VoterVeto final score")
axes[1].set_title("Thermal probability: P(T > 400 °C)")
axes[2].set_title("Combined target (color clipped at global 99th percentile)")
for axis in axes:
    axis.set_axis_off()
fig.suptitle(f"Newberry 3-D slice at elevation {display_z:,.0f} m MSL")
figure_output = os.environ.get("GEOPFA_DEMO_FIGURE_PATH")
if figure_output:
    figure_path = Path(figure_output).expanduser().resolve()
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_path, dpi=240, bbox_inches="tight")
plt.show()

In [ ]:
{
    "target": "P(T > 400 C) in the Newberry 3-D volume",
    "n_grid_cells": int(len(heat_probability)),
    "display_elevation_m_msl": display_z,
    "heat_probability": heat_probability.probability.describe().to_dict(),
    "combined_probability": combined_probability.probability.describe().to_dict(),
    "assessment_scope": (
        "The public thermal mean field is paired with a provenance-bound predictive SD "
        "derived from held-out residual assessment. That assessment supports the "
        "uncertainty construction but is not target-matched 400 C calibration. No "
        "population calibration claim is made for 400 C because no target-matched "
        "labels exist."
    ),
}

## Interpretation and caveat

This is an illustrative, low-data Bayesian prior-predictive result: geoPFA propagates a physically interpretable thermal probability and explicit parameter-prior uncertainty through a real 3-D target volume without pretending that 150 °C proxy labels validate a 400 °C endpoint. Full-grid raw draw blocks are disabled in this demonstration; publication runs require a predeclared Monte Carlo convergence gate and a storage plan for any retained draws. The map is a screening product. It is not proof of a drillable reservoir and has no target-matched population calibration estimate.